# Reverie: Can taste from the past predict the future?
### A temporal-holdout validation (Deep Learning, IE)

The Reverie recommender is trained only on movies up to the year 2000. This notebook asks a different question with a **proper deep learning experiment**: if we learn a person's taste from the movies they rated **before 2020**, can we predict whether they will rate **2020 and later** movies **3 stars or above**?

**Method (all in scope of the course):**
- A **supervised binary classifier** (predict `rated_3plus`), exactly the diabetes ANN pattern: scaled features, `Dense -> Dropout -> Dense -> Dropout -> sigmoid`, `binary_crossentropy`, threshold 0.5.
- A **temporal holdout**: pre-2020 to train, post-2020 hidden as the test set (the "Slicing the Timeline" idea). The test films are never seen during training.
- **Baselines it must beat:** a majority-class baseline, and a content baseline (cosine to a genre taste vector).
- **Evaluation:** confusion matrix and precision/recall, and training vs validation loss curves to watch for overfitting.

**Honest caveats:** one user, a few hundred films, and ratings skew positive (we rate films we expected to like). So accuracy alone is misleading; we lead with precision/recall against the baselines. Genre is a coarse signal, so a modest lift is the realistic, valid outcome.

# STEP 0: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
tf.random.set_seed(42)

ML_GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]

# STEP 1: Load the dataset
Built by `scripts/build_validation_dataset.py` from a Letterboxd export (title, year, rating, `rated_3plus`, and an 18-genre multi-hot). Personal data, kept local.

In [ ]:
df = pd.read_csv("../data/validation_dataset.csv")
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"]).reset_index(drop=True)
df["year"] = df["year"].astype(int)
print(df.shape)
df.head()

# STEP 2: Explore (class balance and the timeline)

In [ ]:
print("Rated 3+ (positive class) share:", df["rated_3plus"].mean().round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(x="rated_3plus", data=df, ax=ax[0]); ax[0].set_title("Label balance (1 = rated 3+)")
sns.histplot(df["year"], bins=30, ax=ax[1]); ax[1].axvline(2020, color="red", ls="--")
ax[1].set_title("Films by release year (red = 2020 split)")
plt.tight_layout()

# STEP 3: The temporal split (no leakage)
Pre-2020 trains the model (with an internal validation split for early stopping). 2020+ is the **test set, locked away** and never seen during training. The scaler is fit on the training data only.

In [ ]:
train_full = df[df["year"] < 2020].copy()
test = df[df["year"] >= 2020].copy()
print(f"train (pre-2020): {len(train_full)}   |   test (2020+): {len(test)}")

tr, val = train_test_split(
    train_full, test_size=0.2, random_state=42, stratify=train_full["rated_3plus"]
)

sc = StandardScaler()
Xtr = sc.fit_transform(tr[ML_GENRES]);   ytr = tr["rated_3plus"].values
Xval = sc.transform(val[ML_GENRES]);     yval = val["rated_3plus"].values
Xte = sc.transform(test[ML_GENRES]);     yte = test["rated_3plus"].values
print("train/val/test:", len(ytr), len(yval), len(yte))

# STEP 4: Baselines to beat

**(a) Majority class** — always predict the most common label.
**(b) Content bridge (cosine)** — taste = mean genre vector of the pre-2020 films rated 3+; score each film by cosine to that taste; threshold tuned on the validation set. Cosine is not taught in the course, so it is used only as a baseline, not as the method.

In [ ]:
# (a) majority-class baseline
majority = int(round(ytr.mean()))
ypred_maj = np.full_like(yte, majority)
print("=== Majority-class baseline ===")
print(classification_report(yte, ypred_maj, zero_division=0))

In [ ]:
# (b) cosine content-bridge baseline
def cos(a, b):
    return float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

liked = train_full[train_full["rated_3plus"] == 1][ML_GENRES].values
taste = liked.mean(axis=0)

val_scores = np.array([cos(r, taste) for r in val[ML_GENRES].values])
thresholds = np.linspace(val_scores.min(), val_scores.max(), 60)
best_th = max(thresholds, key=lambda t: ((val_scores >= t).astype(int) == yval).mean())

test_scores = np.array([cos(r, taste) for r in test[ML_GENRES].values])
ypred_cos = (test_scores >= best_th).astype(int)
print(f"=== Cosine content-bridge baseline (threshold {best_th:.3f}) ===")
print(classification_report(yte, ypred_cos, zero_division=0))

# STEP 5: The neural network (the in-scope method)
The diabetes ANN pattern: stacked Dense layers with ReLU, dropout for regularization, a single sigmoid output, `binary_crossentropy`, and early stopping on the validation loss. Class weights handle the positive-class imbalance.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(ML_GENRES),)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=ytr)
class_weight = {0: float(cw[0]), 1: float(cw[1])}

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True
)
hist = model.fit(
    Xtr, ytr, validation_data=(Xval, yval),
    epochs=300, batch_size=16, class_weight=class_weight,
    callbacks=[early], verbose=0,
)
print(f"stopped after {len(hist.history['loss'])} epochs")

In [ ]:
# Train vs validation loss: our overfitting check
plt.figure(figsize=(6, 3.5))
plt.plot(hist.history["loss"], label="train loss")
plt.plot(hist.history["val_loss"], label="validation loss")
plt.xlabel("epoch"); plt.ylabel("binary cross-entropy"); plt.legend()
plt.title("Training vs validation loss")

In [ ]:
yprob = model.predict(Xte, verbose=0).ravel()
ypred_nn = (yprob >= 0.5).astype(int)
print("=== Neural network (test = 2020+, never seen) ===")
print(classification_report(yte, ypred_nn, zero_division=0))

# STEP 6: Compare on the held-out post-2020 films

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

def row(name, yp):
    return {
        "model": name,
        "accuracy": round(accuracy_score(yte, yp), 3),
        "precision": round(precision_score(yte, yp, zero_division=0), 3),
        "recall": round(recall_score(yte, yp, zero_division=0), 3),
        "f1": round(f1_score(yte, yp, zero_division=0), 3),
    }

summary = pd.DataFrame([
    row("majority baseline", ypred_maj),
    row("cosine content bridge", ypred_cos),
    row("neural network", ypred_nn),
]).set_index("model")
summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, (name, yp) in zip(ax, [("Majority", ypred_maj), ("Cosine", ypred_cos), ("Neural net", ypred_nn)]):
    sns.heatmap(confusion_matrix(yte, yp), annot=True, fmt="d", cbar=False, ax=a)
    a.set_title(name); a.set_xlabel("predicted"); a.set_ylabel("actual")
plt.tight_layout()

# Takeaways

- The **temporal holdout** means every test film was released after the model learned the user's taste, a genuine "never seen during training" test, exactly the discipline the course stresses.
- Lead with **precision and recall**, not accuracy: because most films are rated 3+, a majority baseline already looks "accurate". The real question is whether the neural network's precision/recall beats both baselines.
- If the network beats the cosine bridge and the majority baseline, genre-based taste carries real, generalizable signal about future preferences. If it only ties, the honest finding is that taste is more than genre, which motivates richer features (cast, director, embeddings) as future work.
- **Regularization** (dropout, early stopping on validation loss) keeps the small dataset from overfitting; the loss curves above are the evidence.